# Laboratorio de preparación para la PC1

Este cuaderno permite **comprobar** cálculos de la guía después de resolverlos
manualmente. Utiliza `sympy` para mantener aritmética exacta.

La meta no es memorizar comandos. En cada sección debes anticipar el resultado,
ejecutar la celda y explicar qué significan los números obtenidos.


In [ ]:
import sympy as sp
from sympy import Matrix
sp.init_printing()

def resumen_matriz(A):
    R, pivotes = A.rref()
    return {
        "rref": R,
        "pivotes": pivotes,
        "rango": len(pivotes),
        "nulidad": A.cols - len(pivotes),
        "base_imagen": A.columnspace(),
        "base_nucleo": A.nullspace(),
    }

def clasificar_sistema(A, b):
    rango_A = A.rank()
    rango_aumentada = A.row_join(b).rank()
    if rango_A < rango_aumentada:
        tipo = "incompatible"
    elif rango_A == A.cols:
        tipo = "compatible determinado"
    else:
        tipo = "compatible indeterminado"
    return rango_A, rango_aumentada, tipo

def datos_suma_directa(U, W):
    if U.rows != W.rows:
        raise ValueError("U y W deben estar en el mismo espacio ambiente.")
    if U.rank() != U.cols or W.rank() != W.cols:
        raise ValueError("Las columnas de entrada deben ser bases.")
    B = U.row_join(W)
    dim_inter = U.cols + W.cols - B.rank()
    return {
        "dim_suma": B.rank(),
        "dim_interseccion": dim_inter,
        "es_directa": dim_inter == 0,
        "genera_ambiente": B.rank() == U.rows,
    }

def coords_en_base(x, B):
    if B.rows != B.cols or B.det() == 0:
        raise ValueError("Las columnas no forman una base del espacio.")
    return B.LUsolve(x)

def cambio_base(B, C):
    # P_{B->C}: recibe [x]_B y entrega [x]_C.
    if B.shape != C.shape or B.det() == 0 or C.det() == 0:
        raise ValueError("B y C deben ser matrices de bases compatibles.")
    return C.inv() * B


## 1. Producto interno y norma

Antes de ejecutar, anticipa si el ángulo será agudo, recto u obtuso.


In [ ]:
u = Matrix([1, -2, 2])
v = Matrix([2, 1, 0])

producto = u.dot(v)
norma_u = sp.sqrt(u.dot(u))
norma_v = sp.sqrt(v.dot(v))
coseno = sp.simplify(producto / (norma_u * norma_v))

display(producto, norma_u, norma_v, coseno)
assert abs(producto) <= norma_u * norma_v
assert sp.sqrt((u + v).dot(u + v)) <= norma_u + norma_v


El producto es cero, de modo que los vectores son ortogonales. La
comprobación particular no reemplaza las pruebas generales de las
desigualdades.


## 2. Sistema con parámetro

Para $\lambda\neq\pm1$ hay tres pivotes. En los valores excepcionales debemos
comparar la ecuación que desaparece con su término independiente.


In [ ]:
lam = sp.symbols('lambda', real=True)
A_lam = Matrix([[1, 1, 1],
                [0, lam - 1, 1],
                [0, 0, lam + 1]])
b_lam = Matrix([2, sp.Rational(1, 2), 1])

display(sp.factor(A_lam.det()))
for valor in [-1, 1, 2]:
    A_v = A_lam.subs(lam, valor)
    print(valor, clasificar_sistema(A_v, b_lam))
    display(A_v.row_join(b_lam).rref()[0])


## 3. Rango, imagen, núcleo y compatibilidad

Los pivotes se identifican en la RREF, pero la base de la imagen se toma de las
columnas de la matriz original.


In [ ]:
A = Matrix([
    [1, 2, 0, 1, -1, 3],
    [2, 4, 1, 3,  0, 7],
    [1, 2, 1, 2,  1, 4],
    [3, 6, 1, 4, -1, 10],
])

info = resumen_matriz(A)
display(info["rref"])
print("pivotes:", info["pivotes"])
print("rango:", info["rango"], "nulidad:", info["nulidad"])
display(*info["base_imagen"])
display(*info["base_nucleo"])

assert info["rango"] + info["nulidad"] == A.cols
assert all(A * z == sp.zeros(A.rows, 1) for z in info["base_nucleo"])


In [ ]:
b = Matrix([1, 3, 2, 4])
b_incompatible = Matrix([1, 3, 2, 5])

print("b:", clasificar_sistema(A, b))
display(sp.linsolve((A, b)))
print("b':", clasificar_sistema(A, b_incompatible))
display(sp.linsolve((A, b_incompatible)))


El primer conjunto solución es una traslación de $\ker(A)$ y tiene
cuatro parámetros. El segundo es vacío porque el rango de la matriz aumentada
supera al rango de $A$.


## 4. Determinante e invertibilidad

La factorización del determinante detecta los valores singulares. Después se
comprueban mediante el rango y el núcleo.


In [ ]:
D = Matrix([[lam, 1, 0],
            [1, lam, 0],
            [0, 0, lam + 2]])
det_D = sp.factor(D.det())
display(det_D)

for valor in [-2, -1, 1]:
    D_v = D.subs(lam, valor)
    print("lambda =", valor, "rango =", D_v.rank())
    display(D_v.nullspace())
    assert D_v.det() == 0


In [ ]:
G = Matrix([[1, 2, 0], [0, 1, 1], [2, 3, 1]])
G_inv = G.inv()
display(G.det(), G_inv)
assert G * G_inv == sp.eye(3)


## 5. Plano afín

Para $H=\{x:x_1+2x_2-x_3=3\}$, una solución particular fija el punto de apoyo
y el núcleo proporciona las direcciones.


In [ ]:
L = Matrix([[1, 2, -1]])
d = Matrix([3])
x0 = Matrix([3, 0, 0])
direcciones = L.nullspace()

display(x0, *direcciones)
assert L * x0 == d
assert all(L * z == sp.zeros(1, 1) for z in direcciones)


## 6. Suma directa y descomposición

Si la matriz que reúne las bases es invertible, cada vector de $\mathbb R^4$
posee una descomposición única.


In [ ]:
U = Matrix.hstack(Matrix([1, 0, 1, 0]), Matrix([0, 1, 0, 1]))
W = Matrix.hstack(Matrix([1, 0, -1, 0]), Matrix([0, 1, 0, -1]))
B_UW = U.row_join(W)

display(datos_suma_directa(U, W))
display(B_UW.det())

x = Matrix([3, 1, -1, 5])
coef = B_UW.LUsolve(x)
u_parte = U * coef[:U.cols, :]
w_parte = W * coef[U.cols:, :]
display(coef, u_parte, w_parte)
assert u_parte + w_parte == x


## 7. Coordenadas y cambio de base

Verificaremos el recorrido

$$[x]_{\mathcal B}\xrightarrow{M_{\mathcal B}}x
\xrightarrow{M_{\mathcal C}^{-1}}[x]_{\mathcal C}.$$


In [ ]:
B = Matrix([[1, 1], [1, -1]])
C = Matrix([[2, 1], [0, 1]])
x = Matrix([5, 1])

P_BC = cambio_base(B, C)
x_B = coords_en_base(x, B)
x_C = coords_en_base(x, C)

display(P_BC, x_B, x_C)
assert P_BC * x_B == x_C
assert cambio_base(C, B) * P_BC == sp.eye(2)


In [ ]:
A_T = Matrix([[2, 1], [1, 0]])
T_CB = C.inv() * A_T * B
r, s = sp.symbols('r s')
c = Matrix([r, s])

display(T_CB)
assert sp.simplify(C.inv() * A_T * B * c - T_CB * c) == sp.zeros(2, 1)


## 8. Comprobación numérica y tolerancias

`numpy` trabaja normalmente con punto flotante. Una cantidad matemáticamente
nula puede aparecer como un número muy pequeño; por ello se usa una tolerancia.


In [ ]:
import numpy as np

A_num = np.array(A.tolist(), dtype=float)
rango_num = np.linalg.matrix_rank(A_num, tol=1e-10)
print("rango numérico:", rango_num)
assert rango_num == A.rank()


## 9. Retos para completar sin mirar la clave

1. Escribe una función que devuelva una base explícita de $U\cap W$.
2. Modifica el sistema paramétrico para que sus dos valores excepcionales sean
   compatibles indeterminados.
3. Construye tres bases y verifica la composición de cambios de base.
4. Para la matriz del simulacro, genera bases de imagen y núcleo y explica sus
   dimensiones.
5. Implementa Gauss–Jordan con aritmética exacta y registra cada operación.
